In [1]:
!pip install fastapi uvicorn[standard] pydantic httpx nest-asyncio -q

# Phase 4 — Deploy & Ship
## Day 25: FastAPI — Wrap Your RAG Pipeline in a Real API

**Goal:** Turn the Indian Legal RAG chatbot from a Gradio UI into a proper REST API
with documented endpoints, type-safe request/response models, and Swagger UI.

**Today's endpoints:**
- `GET  /health`   — status check
- `POST /chat`     — question → grounded answer + sources
- `POST /predict`  — text → sentiment label + confidence

**Stack:** FastAPI + Pydantic + Uvicorn

## Part 1: What Makes FastAPI Different

FastAPI sits on top of two libraries:
- **Starlette** — handles HTTP routing, requests, responses (the web layer)
- **Pydantic** — handles data validation (the safety layer)

The combination means: you define *what* valid data looks like,
and FastAPI enforces it *before* your code runs.

Compare Flask (manual everything) vs FastAPI (declarative, type-driven).

In [2]:
from pydantic import BaseModel
from typing import Optional, List

class ChatRequest(BaseModel):
    question: str
    session_id: Optional[str] = None

class SourceChunk(BaseModel):
    content: str
    source: str
    page: Optional[int] = None

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]
    session_id: str

# ── Valid input ───────────────────────────────────────────────────────────────
valid = ChatRequest(question="What is BNS Section 103?")
print("Valid:", valid)
print("question type:", type(valid.question))
print("session_id:", valid.session_id)

print()

# ── Pydantic v2: int → str is NOT coerced (strict by default for str) ─────────
print("--- Pydantic v2 strict str behavior ---")
try:
    coerced = ChatRequest(question=42)
except Exception as e:
    print(f"int → str: REJECTED ({type(e).__name__})")
    print("  Reason: Pydantic v2 does NOT silently coerce int to str")

print()

# ── What Pydantic v2 DOES coerce ──────────────────────────────────────────────
print("--- What v2 DOES coerce ---")

class FlexibleModel(BaseModel):
    count: int      # str "5" → int 5 : YES coerced
    score: float    # int 3 → float 3.0 : YES coerced

f1 = FlexibleModel(count="5", score=3)  # both coerced silently
print(f"str '5' → int: {f1.count}  (type: {type(f1.count).__name__})")
print(f"int 3 → float: {f1.score}  (type: {type(f1.score).__name__})")

print()

# ── opt-in coercion for str using model_config ─────────────────────────────────
print("--- Opt-in str coercion (if you explicitly want it) ---")
from pydantic import ConfigDict

class CoercingModel(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)
    question: str

c = CoercingModel(question=42)
print(f"int 42 → str with coerce_numbers_to_str=True: '{c.question}'")

print()

# ── Missing required field — still caught ─────────────────────────────────────
print("--- Missing required field ---")
try:
    broken = ChatRequest()
except Exception as e:
    print(f"Missing field: REJECTED ({type(e).__name__})")
    print(f"  Detail: {e.errors()[0]['msg']}")

Valid: question='What is BNS Section 103?' session_id=None
question type: <class 'str'>
session_id: None

--- Pydantic v2 strict str behavior ---
int → str: REJECTED (ValidationError)
  Reason: Pydantic v2 does NOT silently coerce int to str

--- What v2 DOES coerce ---
str '5' → int: 5  (type: int)
int 3 → float: 3.0  (type: float)

--- Opt-in str coercion (if you explicitly want it) ---
int 42 → str with coerce_numbers_to_str=True: '42'

--- Missing required field ---
Missing field: REJECTED (ValidationError)
  Detail: Field required


## Part 2: Build the FastAPI App — Step by Step

Three layers:
1. **App instance** — the FastAPI object with metadata
2. **Startup event** — load the RAG pipeline *once*, store it in app state
3. **Endpoints** — thin functions that call the pipeline and return typed responses

The startup event is critical. Loading a model takes seconds.
You load it once when the server starts — not on every request.

In [3]:
# %%writefile app/main.py
# We write this to a file — this IS the production code, not just a demo

app_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import os
import uuid

# ── App Metadata ─────────────────────────────────────────────────────────────
app = FastAPI(
    title="Indian Legal RAG API",
    description="""
    REST API wrapping a Retrieval-Augmented Generation pipeline 
    over the Bharatiya Nyaya Sanhita (BNS) 2023.
    
    - POST /chat: Ask a legal question, get a grounded answer with source citations
    - POST /predict: Run sentiment classification on any text
    - GET  /health: Check if the API is alive
    """,
    version="1.0.0",
    contact={"name": "Faisal Imam", "url": "https://github.com/faisalimam1"},
)

# ── Pydantic Models — Request/Response Contracts ──────────────────────────────
class ChatRequest(BaseModel):
    question: str
    session_id: Optional[str] = None       # client passes this to maintain context

class SourceChunk(BaseModel):
    content: str                            # the actual retrieved text chunk
    source: str                             # document name / section
    relevance_score: Optional[float] = None

class ChatResponse(BaseModel):
    answer: str
    sources: List[SourceChunk]
    session_id: str                         # always returned so client can track

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    label: str                              # "POSITIVE" or "NEGATIVE"
    confidence: float                       # 0.0 → 1.0
    model_used: str

class HealthResponse(BaseModel):
    status: str
    pipeline_loaded: bool
    version: str

# ── In-memory session store (replace with Redis in production) ────────────────
# Maps session_id → list of (question, answer) tuples
session_store: dict = {}

# ── Pipeline placeholder (replace with your actual RAG pipeline) ──────────────
# In production: load ChromaDB, BM25 index, Groq client at startup
# For this demo: we return structured mock responses

def get_rag_answer(question: str, session_id: str) -> tuple[str, List[dict]]:
    """
    In production this calls your Phase 3 RAG pipeline:
        1. Hybrid retrieval (BM25 + semantic, RRF fusion)
        2. Context assembly
        3. Groq LLM call with retrieved context
        4. Return answer + source chunks
    
    For notebook demo: returns structured mock data.
    """
    history = session_store.get(session_id, [])
    
    # Mock response — replace this block with actual pipeline call
    mock_answer = (
        f"Based on the Bharatiya Nyaya Sanhita 2023, regarding '{question}': "
        f"[This is where your LangChain LCEL chain returns the grounded answer. "
        f"Context from {len(history)} prior turns available.]"
    )
    mock_sources = [
        {"content": "BNS Section 103 — Murder...", "source": "BNS_2023.pdf", "relevance_score": 0.91},
        {"content": "BNS Section 104 — Culpable homicide...", "source": "BNS_2023.pdf", "relevance_score": 0.87},
    ]
    
    # Store in session
    session_store[session_id] = history + [(question, mock_answer)]
    
    return mock_answer, mock_sources


def get_sentiment(text: str) -> dict:
    """
    In production: loads your fine-tuned BERT from HF Hub.
    For demo: returns mock sentiment.
    """
    return {
        "label": "POSITIVE" if len(text) % 2 == 0 else "NEGATIVE",
        "confidence": 0.94,
        "model_used": "faisalimam19/bert-imdb-sentiment"
    }


# ── Endpoints ──────────────────────────────────────────────────────────────────

@app.get("/health", response_model=HealthResponse)
def health_check():
    """Check if the API is alive and the pipeline is loaded."""
    return HealthResponse(
        status="healthy",
        pipeline_loaded=True,
        version="1.0.0"
    )


@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Ask a legal question. Returns a grounded answer with source citations.
    
    - Pass session_id to maintain conversation context across turns.
    - If session_id is None, a new session is created and returned.
    """
    if not request.question.strip():
        raise HTTPException(status_code=422, detail="question cannot be empty")
    
    # Generate session ID if client didn't provide one
    session_id = request.session_id or str(uuid.uuid4())
    
    try:
        answer, raw_sources = get_rag_answer(request.question, session_id)
    except Exception as e:
        # Never expose internal errors to the client
        raise HTTPException(status_code=500, detail="RAG pipeline error. Check server logs.")
    
    sources = [SourceChunk(**s) for s in raw_sources]
    
    return ChatResponse(
        answer=answer,
        sources=sources,
        session_id=session_id
    )


@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    """
    Run sentiment classification on input text.
    Uses the fine-tuned BERT model (F1: 0.921 on IMDB).
    """
    if len(request.text.strip()) < 3:
        raise HTTPException(status_code=422, detail="text too short for classification")
    
    try:
        result = get_sentiment(request.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail="Model inference failed. Check server logs.")
    
    return PredictResponse(**result)
'''

# Write to file
import os
os.makedirs("app", exist_ok=True)

with open("app/main.py", "w") as f:
    f.write(app_code)

print("✅ app/main.py written")
print(f"   Lines: {len(app_code.splitlines())}")

✅ app/main.py written
   Lines: 155


## Part 3: Run the Server and Test It

**How FastAPI runs:**
- `uvicorn` is the ASGI server — it receives HTTP requests and passes them to FastAPI
- `app.main:app` → "in the file app/main.py, find the object called app"
- `--reload` → restart server when code changes (dev mode only, never in production)

**In production you'd run:**
```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

**Here in the notebook**, we run it in a background thread so we can 
test it from the same kernel using the `requests` library.

In [4]:
import threading
import time
import nest_asyncio

# FastAPI uses async — Kaggle's kernel is also async — this lets them coexist
nest_asyncio.apply()

def run_server():
    import uvicorn
    # Import the app from the file we just wrote
    import sys
    sys.path.insert(0, ".")
    exec(open("app/main.py").read(), globals())
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Give the server 2 seconds to start
time.sleep(2)
print("✅ Server started at http://localhost:8000")
print("   Swagger UI: http://localhost:8000/docs")

/usr/local/lib/python3.12/dist-packages/uvicorn/server.py:75: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
Exception in thread Thread-5 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_58/3793162754.py", line 14, in run_server
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 606, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword 

✅ Server started at http://localhost:8000
   Swagger UI: http://localhost:8000/docs


In [5]:
# ── Cell 1: Start server — no nest_asyncio needed ────────────────────────────
import asyncio
import uvicorn
import threading
import time
import requests
import json
import sys

sys.path.insert(0, ".")
from app.main import app

def run_server():
    # Create a fresh event loop in this thread — completely isolated from Kaggle's kernel loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)

try:
    ping = requests.get("http://localhost:8000/health", timeout=3)
    print(f"✅ Server is UP — {ping.status_code}")
    print(f"   {ping.json()}")
except Exception as e:
    print(f"❌ Still failed: {e}")
    print("   → Try Option B below")

✅ Server is UP — 200
   {'status': 'healthy', 'pipeline_loaded': True, 'version': '1.0.0'}


In [6]:
# ── All endpoint tests ────────────────────────────────────────────────────────
import requests
import json

BASE_URL = "http://localhost:8000"

# ── TEST 1: Health ────────────────────────────────────────────────────────────
print("=" * 55)
print("TEST 1: GET /health")
print("=" * 55)
r = requests.get(f"{BASE_URL}/health")
print(f"Status : {r.status_code}")
print(f"Body   : {json.dumps(r.json(), indent=2)}")

# ── TEST 2: POST /chat — new session ─────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 2: POST /chat — Turn 1 (new session)")
print("=" * 55)
r = requests.post(f"{BASE_URL}/chat",
                  json={"question": "What is the punishment for murder under BNS 2023?"})
print(f"Status     : {r.status_code}")
data = r.json()
session_id = data["session_id"]
print(f"Session ID : {session_id}")
print(f"Answer     : {data['answer'][:120]}...")
print(f"Sources    : {len(data['sources'])} chunks")
for i, s in enumerate(data["sources"]):
    print(f"  [{i+1}] {s['source']} — relevance: {s['relevance_score']}")

# ── TEST 3: POST /chat — same session (memory test) ──────────────────────────
print("\n" + "=" * 55)
print("TEST 3: POST /chat — Turn 2 (same session = memory)")
print("=" * 55)
r = requests.post(f"{BASE_URL}/chat",
                  json={"question": "What about attempt to murder?",
                        "session_id": session_id})
data2 = r.json()
print(f"Status     : {r.status_code}")
print(f"Session ID : {data2['session_id']}")
print(f"Same as T1 : {data2['session_id'] == session_id}")
print(f"Answer     : {data2['answer'][:120]}...")

# ── TEST 4: POST /predict ─────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 4: POST /predict — Sentiment Classification")
print("=" * 55)
for text in [
    "This movie was absolutely brilliant, I loved every minute.",
    "Terrible film, complete waste of time.",
]:
    r = requests.post(f"{BASE_URL}/predict", json={"text": text})
    res = r.json()
    print(f"\nInput : {text}")
    print(f"Label : {res['label']}  |  Confidence: {res['confidence']:.2%}")
    print(f"Model : {res['model_used']}")

# ── TEST 5: Validation — bad inputs ──────────────────────────────────────────
print("\n" + "=" * 55)
print("TEST 5: Validation — bad inputs caught automatically")
print("=" * 55)

r = requests.post(f"{BASE_URL}/chat", json={})
print(f"\n[A] Missing 'question' → Status: {r.status_code}")
print(f"    Error: {r.json()['detail'][0]['msg']}")

r = requests.post(f"{BASE_URL}/chat", json={"question": ""})
print(f"\n[B] Empty question → Status: {r.status_code}")
print(f"    Error: {r.json()['detail']}")

r = requests.post(f"{BASE_URL}/predict", json={"text": "hi"})
print(f"\n[C] Text too short → Status: {r.status_code}")
print(f"    Error: {r.json()['detail']}")

TEST 1: GET /health
Status : 200
Body   : {
  "status": "healthy",
  "pipeline_loaded": true,
  "version": "1.0.0"
}

TEST 2: POST /chat — Turn 1 (new session)
Status     : 200
Session ID : 7b16cd3f-3471-4f74-97ed-aa35c15c9ae5
Answer     : Based on the Bharatiya Nyaya Sanhita 2023, regarding 'What is the punishment for murder under BNS 2023?': [This is where...
Sources    : 2 chunks
  [1] BNS_2023.pdf — relevance: 0.91
  [2] BNS_2023.pdf — relevance: 0.87

TEST 3: POST /chat — Turn 2 (same session = memory)
Status     : 200
Session ID : 7b16cd3f-3471-4f74-97ed-aa35c15c9ae5
Same as T1 : True
Answer     : Based on the Bharatiya Nyaya Sanhita 2023, regarding 'What about attempt to murder?': [This is where your LangChain LCEL...

TEST 4: POST /predict — Sentiment Classification

Input : This movie was absolutely brilliant, I loved every minute.
Label : POSITIVE  |  Confidence: 94.00%
Model : faisalimam19/bert-imdb-sentiment

Input : Terrible film, complete waste of time.
Label : POSITIVE  

## Part 4: The Swagger UI — Your Free API Documentation

**Go to http://localhost:8000/docs in your browser.**

What you'll see:
- Every endpoint listed with its HTTP method
- Click "Try it out" → fill in the request body → Execute
- See the actual HTTP response, status code, and response schema
- All of this was generated *automatically* from your Pydantic models

This is what you show a hiring manager or a potential collaborator.
Not a notebook — a real, documented, interactive API.

**http://localhost:8000/redoc** — alternative docs, cleaner for reading.
**http://localhost:8000/openapi.json** — the raw schema (machine-readable).

## Part 5: How to Connect Your Actual RAG Pipeline

The `get_rag_answer()` function above uses mock data.
To connect your Phase 3 pipeline, replace it with:

```python
# At startup — load once
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
import chromadb

# These are loaded ONCE when the server starts
chroma_client = chromadb.PersistentClient(path="./chroma_bns_db")
retriever = ... # your hybrid BM25 + semantic retriever
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.environ["GROQ_API_KEY"])
chain = ... # your LCEL chain

# In the endpoint — called per request
def get_rag_answer(question, session_id):
    history = session_store.get(session_id, [])
    result = chain.invoke({"question": question, "chat_history": history})
    return result["answer"], result["sources"]
```

Key principle: the ChromaDB client, retriever, LLM, and chain are 
instantiated ONCE at startup. The endpoint function only calls `.invoke()`.

In [7]:
requirements = """fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0.0
langchain>=0.1.0
langchain-groq>=0.1.0
langchain-chroma>=0.1.0
chromadb>=0.4.0
sentence-transformers>=2.2.2
rank-bm25>=0.2.2
python-dotenv>=1.0.0
"""

with open("app/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt written")
print(requirements)

✅ requirements.txt written
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0.0
langchain>=0.1.0
langchain-groq>=0.1.0
langchain-chroma>=0.1.0
chromadb>=0.4.0
sentence-transformers>=2.2.2
rank-bm25>=0.2.2
python-dotenv>=1.0.0



## Day 25 Summary

**What we built:**
- A production-structured FastAPI app with 3 endpoints
- Pydantic request/response models — type-safe contracts
- Session management for multi-turn conversation context
- Proper HTTP error handling with HTTPException
- Auto-generated Swagger UI documentation

**Key numbers to remember:**
- `200` — success
- `422` — validation error (Pydantic caught bad input)
- `500` — internal server error (your pipeline crashed)

**The pattern you'll use forever:**
1. Define Pydantic models (what's valid in, what's valid out)
2. Load heavy resources at startup (not per request)
3. Keep endpoints thin — they call other functions, they don't do the work
4. Never expose raw exceptions — always HTTPException with a safe message



## Day 26: Polished Gradio UI — Calling FastAPI over HTTP

The UI is a client of the API. It knows nothing about RAG, ChromaDB, or BERT.
It only speaks JSON over HTTP.

Architecture:
  Gradio ChatInterface → POST /chat → FastAPI → RAG pipeline → JSON → Gradio displays it

Stack: Gradio + requests

## Day 26: Polished Gradio UI — Calling FastAPI over HTTP

The UI is a client of the API. It knows nothing about RAG, ChromaDB, or BERT.
It only speaks JSON over HTTP.

Architecture:
  Gradio → POST /chat → FastAPI → RAG pipeline → JSON → Gradio displays it

Key rule: FastAPI must start as a subprocess BEFORE importing Gradio.
Gradio applies nest_asyncio internally on import — which breaks uvicorn
if they share the same Python process. Subprocess = separate process = no conflict.

In [1]:
import subprocess
import sys
import time
import requests

# Kill anything already on port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(1)

proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)

if proc.poll() is not None:
    print("❌ uvicorn crashed")
    print(proc.stderr.read().decode())
else:
    try:
        ping = requests.get("http://localhost:8000/health", timeout=3)
        print(f"✅ FastAPI UP — {ping.json()['status']}")
    except Exception as e:
        print(f"❌ Not responding: {e}")
        print(proc.stderr.read().decode())

✅ FastAPI UP — healthy


## Part 1: Why the UI is just a client

The Gradio function does exactly one thing: sends an HTTP POST and displays the result.
It contains zero business logic — no RAG, no embeddings, no LLM calls.

This means:
- You can replace Gradio with React tomorrow — the API doesn't change
- You can replace the RAG pipeline with a different model — the UI doesn't change
- Two engineers can work on them simultaneously without conflicts

That's the separation of concerns that makes production systems maintainable.

## Part 2: gr.State — How Multi-Turn Memory Works in Gradio

`gr.State` stores a value client-side across interactions.
Each browser tab gets its own independent State — so each user gets their own session_id.

Without State: every message would start a new session, losing conversation history.
With State: session_id flows through every interaction, giving the API context.

In [2]:
import gradio as gr
import requests

API_URL = "http://localhost:8000"

# ── Chat function — the entire UI logic is here ───────────────────────────────
def chat_fn(message, history, session_id):
    if not message.strip():
        return history, "", session_id

    payload = {"question": message}
    if session_id:
        payload["session_id"] = session_id  # pass existing session for memory

    try:
        r = requests.post(f"{API_URL}/chat", json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()

        # Format sources below the answer
        source_text = ""
        if data["sources"]:
            source_text = "\n\n**Sources retrieved:**\n"
            for i, s in enumerate(data["sources"], 1):
                score = f"{s['relevance_score']:.2f}" if s.get("relevance_score") else "N/A"
                source_text += f"[{i}] {s['source']} — relevance: {score}\n"
                source_text += f"    *{s['content'][:100]}...*\n"

        history.append((message, data["answer"] + source_text))
        return history, "", data["session_id"]  # return new session_id from API

    except requests.exceptions.ConnectionError:
        history.append((message, "⚠️ Cannot reach the API. Check that FastAPI is running."))
        return history, "", session_id
    except requests.exceptions.Timeout:
        history.append((message, "⚠️ Request timed out. Try again."))
        return history, "", session_id
    except Exception as e:
        history.append((message, f"⚠️ Unexpected error: {type(e).__name__}"))
        return history, "", session_id


# ── Sentiment function ────────────────────────────────────────────────────────
def predict_fn(text):
    if len(text.strip()) < 3:
        return "⚠️ Text too short. Enter at least a few words."
    try:
        r = requests.post(f"{API_URL}/predict", json={"text": text}, timeout=10)
        r.raise_for_status()
        data = r.json()
        emoji = "🟢" if data["label"] == "POSITIVE" else "🔴"
        return (
            f"{emoji} **{data['label']}**\n"
            f"Confidence: {data['confidence']:.2%}\n"
            f"Model: `{data['model_used']}`"
        )
    except Exception as e:
        return f"⚠️ Error: {str(e)}"


# ── Two-tab UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Indian Legal RAG API Demo", theme=gr.themes.Soft()) as full_app:

    gr.Markdown("""
    # ⚖️ Indian Legal RAG — API Demo
    *FastAPI backend · LangChain LCEL · ChromaDB · BM25 Hybrid Search · Groq (Llama 3.3 70B)*
    """)

    with gr.Tab("💬 Legal Chatbot"):

        session_state = gr.State(value=None)  # persists session_id across turns

        chatbot = gr.Chatbot(
            height=420,
            show_copy_button=True,
            bubble_full_width=False,
            label="Conversation"
        )

        with gr.Row():
            msg = gr.Textbox(
                placeholder="Ask a legal question — e.g. What is punishment for murder under BNS?",
                label="Your Question",
                scale=4,
                autofocus=True
            )
            submit = gr.Button("Ask", variant="primary", scale=1)

        clear = gr.Button("Clear Conversation", variant="secondary")

        gr.Examples(
            examples=[
                "What is the punishment for murder under BNS 2023?",
                "Define culpable homicide and how is it different from murder?",
                "What are the rights of an arrested person?",
                "Explain the concept of self-defence under BNS.",
            ],
            inputs=msg,
            label="Example Questions"
        )

        # Wire interactions
        submit.click(chat_fn, [msg, chatbot, session_state], [chatbot, msg, session_state])
        msg.submit(chat_fn, [msg, chatbot, session_state], [chatbot, msg, session_state])
        clear.click(lambda: ([], "", None), outputs=[chatbot, msg, session_state])

    with gr.Tab("🎭 Sentiment Classifier"):

        gr.Markdown("""
        **Fine-tuned BERT** (F1: 0.921 on IMDB) served via `POST /predict`
        
        Enter any text — the model classifies it as POSITIVE or NEGATIVE.
        """)

        text_input = gr.Textbox(
            placeholder="Enter any text to classify...",
            label="Input Text",
            lines=4
        )
        predict_btn = gr.Button("Classify", variant="primary")
        result_output = gr.Markdown(label="Result")

        gr.Examples(
            examples=[
                "This movie was absolutely brilliant, I loved every minute.",
                "Terrible film, complete waste of time.",
                "The judgment was fair and well-reasoned.",
            ],
            inputs=text_input,
            label="Example Inputs"
        )

        predict_btn.click(predict_fn, inputs=text_input, outputs=result_output)
        text_input.submit(predict_fn, inputs=text_input, outputs=result_output)

    gr.Markdown("""
    ---
    Built by [Faisal Imam](https://github.com/faisalimam1) · 
    30-Day AI Engineer Roadmap · Day 26/30
    """)

full_app.launch(share=True, show_error=True)

/tmp/ipykernel_113/417707368.py:62: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Indian Legal RAG API Demo", theme=gr.themes.Soft()) as full_app:
/tmp/ipykernel_113/417707368.py:73: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_113/417707368.py:73: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_113/417707368.py:73: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://444d545d6274629471.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Day 27: Docker — Containerise the FastAPI + RAG App

Goal: Write production-ready Docker artifacts for the Indian Legal RAG API.

Files we'll create:
- `Dockerfile` — recipe for building the app image
- `.dockerignore` — exclude unnecessary files from the image
- `docker-compose.yml` — orchestrate FastAPI + ChromaDB together
- `app/.env.example` — document required environment variables

These files go to Render on Day 28 — Render reads the Dockerfile
and builds + deploys automatically on every git push.

In [3]:
import os

dockerignore = """# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
.Python
*.egg-info/
dist/
build/

# Virtual environments
.env
.venv
env/
venv/

# Jupyter
.ipynb_checkpoints/
*.ipynb

# Git
.git/
.gitignore

# OS
.DS_Store
Thumbs.db

# Secrets — never in the image
*.env
.env.*
!.env.example

# Large data files — mount as volumes instead
*.pdf
chroma_db/
"""

with open("app/.dockerignore", "w") as f:
    f.write(dockerignore)

# Also write it at root level
with open(".dockerignore", "w") as f:
    f.write(dockerignore)

print("✅ .dockerignore written")
print()
print("Key exclusions:")
print("  __pycache__/    — Python bytecode, rebuilt inside container")
print("  .git/           — version control history, not needed at runtime")
print("  *.ipynb         — notebooks are dev artifacts, not production code")
print("  .env files      — secrets never go in the image")
print("  chroma_db/      — vector store mounted as a volume, not baked in")

✅ .dockerignore written

Key exclusions:
  __pycache__/    — Python bytecode, rebuilt inside container
  .git/           — version control history, not needed at runtime
  *.ipynb         — notebooks are dev artifacts, not production code
  .env files      — secrets never go in the image
  chroma_db/      — vector store mounted as a volume, not baked in


In [4]:
dockerfile = """# ── Stage: Base Image ────────────────────────────────────────────────────────
# python:3.11-slim = official Python 3.11 on Debian slim (~130MB vs ~1GB full)
FROM python:3.11-slim

# ── System dependencies ───────────────────────────────────────────────────────
# build-essential: needed to compile some Python packages (e.g. chromadb)
# curl: for health checks
RUN apt-get update && apt-get install -y \\
    build-essential \\
    curl \\
    && rm -rf /var/lib/apt/lists/*

# ── Working directory ─────────────────────────────────────────────────────────
WORKDIR /app

# ── Install Python dependencies (cached layer) ────────────────────────────────
# Copy requirements FIRST — before copying code.
# If code changes but requirements don't, Docker reuses this cached layer.
# Build time: 3 min (first) → 10 sec (subsequent builds with same requirements)
COPY app/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy application code ─────────────────────────────────────────────────────
# This layer changes on every code update — but pip layer above is cached
COPY app/ .

# ── Environment variables ─────────────────────────────────────────────────────
# Never hardcode secrets here — pass them at runtime via:
#   docker run -e GROQ_API_KEY=... 
#   or docker-compose environment section
#   or Render's environment variable dashboard
ENV PYTHONUNBUFFERED=1
ENV PYTHONDONTWRITEBYTECODE=1

# ── Port documentation ────────────────────────────────────────────────────────
# EXPOSE is a label — actual port binding happens at docker run -p 8000:8000
EXPOSE 8000

# ── Health check ──────────────────────────────────────────────────────────────
# Docker checks this every 30s — marks container unhealthy if it fails 3 times
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

# ── Start command ─────────────────────────────────────────────────────────────
# --host 0.0.0.0: bind to all interfaces (required — default localhost is unreachable outside container)
# --workers 1: single worker (enough for demo; increase for production)
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
"""

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print("✅ Dockerfile written")
print(f"   Lines: {len(dockerfile.splitlines())}")
print()
print("Layer order (most stable → least stable):")
print("  1. FROM python:3.11-slim     ← never changes")
print("  2. RUN apt-get install       ← rarely changes")
print("  3. COPY requirements.txt     ← changes when deps change")
print("  4. RUN pip install           ← cached if requirements unchanged")
print("  5. COPY app/                 ← changes on every code update")

✅ Dockerfile written
   Lines: 47

Layer order (most stable → least stable):
  1. FROM python:3.11-slim     ← never changes
  2. RUN apt-get install       ← rarely changes
  3. COPY requirements.txt     ← changes when deps change
  4. RUN pip install           ← cached if requirements unchanged
  5. COPY app/                 ← changes on every code update


## Layer Caching — The Most Important Docker Optimization

Docker builds images in layers. Each instruction = one layer.
Layers are cached. If a layer's inputs haven't changed, Docker skips rebuilding it.

The rule: **put things that change least at the top, most at the bottom.**

Wrong order (slow):COPY . .                    ← code changes → cache invalidated

RUN pip install -r req.txt  ← runs every time even if requirements unchanged
Right order (fast):
COPY requirements.txt .     ← only changes when deps change

RUN pip install -r req.txt  ← cached most of the time ✅

COPY . .                    ← code changes here, below the pip layer
First build: 3-4 minutes.
Every subsequent build (code change only): ~10 seconds.

In [5]:
compose = """version: '3.8'

services:

  # ── FastAPI Application ─────────────────────────────────────────────────────
  api:
    build: .                          # build from Dockerfile in current directory
    container_name: legal_rag_api
    ports:
      - "8000:8000"                   # host:container — localhost:8000 → container:8000
    environment:
      - GROQ_API_KEY=${GROQ_API_KEY}  # passed from host .env file — never hardcoded
      - CHROMA_HOST=chromadb          # service name = hostname inside Docker network
      - CHROMA_PORT=8001
    depends_on:
      chromadb:
        condition: service_healthy    # wait for chromadb health check before starting api
    volumes:
      - ./app:/app                    # mount code for development (remove in production)
    restart: unless-stopped           # auto-restart on crash
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # ── ChromaDB Vector Store ───────────────────────────────────────────────────
  chromadb:
    image: chromadb/chroma:latest     # official ChromaDB image — no need to build this
    container_name: legal_rag_chromadb
    ports:
      - "8001:8000"                   # chromadb runs on 8000 inside, exposed as 8001 on host
    volumes:
      - chroma_data:/chroma/chroma    # named volume — data persists across container restarts
    environment:
      - IS_PERSISTENT=TRUE            # enable persistent storage
      - ANONYMIZED_TELEMETRY=FALSE
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/api/v1/heartbeat"]
      interval: 30s
      timeout: 10s
      retries: 5
      start_period: 20s

# ── Named Volumes ─────────────────────────────────────────────────────────────
# Named volumes persist data even when containers are stopped/removed
# chroma_data survives: docker-compose down (but not: docker-compose down -v)
volumes:
  chroma_data:
"""

with open("docker-compose.yml", "w") as f:
    f.write(compose)

print("✅ docker-compose.yml written")
print()
print("Services defined:")
print("  api       — FastAPI app, built from Dockerfile")
print("  chromadb  — ChromaDB vector store, official image")
print()
print("Key concepts in this file:")
print("  depends_on    — api waits for chromadb to be healthy before starting")
print("  volumes       — chroma_data persists across restarts")
print("  environment   — secrets from host .env, never hardcoded")
print("  healthcheck   — Docker monitors both services automatically")

✅ docker-compose.yml written

Services defined:
  api       — FastAPI app, built from Dockerfile
  chromadb  — ChromaDB vector store, official image

Key concepts in this file:
  depends_on    — api waits for chromadb to be healthy before starting
  volumes       — chroma_data persists across restarts
  environment   — secrets from host .env, never hardcoded
  healthcheck   — Docker monitors both services automatically


In [6]:
import os

files = {
    "Dockerfile": "Dockerfile",
    ".dockerignore": ".dockerignore",
    "docker-compose.yml": "docker-compose.yml",
    "app/.env.example": "app/.env.example",
    "app/main.py": "app/main.py",
    "app/requirements.txt": "app/requirements.txt",
}

print("=" * 55)
print("Docker artifact verification")
print("=" * 55)

all_good = True
for name, path in files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "✅" if exists else "❌"
    print(f"{status} {name:<30} {size:>5} bytes")
    if not exists:
        all_good = False

print()
if all_good:
    print("✅ All Docker artifacts present")
    print("   Ready for Day 28 — Render deployment")
else:
    print("❌ Some files missing — re-run the cells above")

print()
print("What Render does with these files on Day 28:")
print("  1. Reads your Dockerfile")
print("  2. Runs docker build automatically")
print("  3. Runs docker run with your environment variables")
print("  4. Gives you a public URL")
print("  5. Auto-redeploys on every git push to main")

Docker artifact verification
✅ Dockerfile                      3245 bytes
✅ .dockerignore                    339 bytes
✅ docker-compose.yml              2303 bytes
❌ app/.env.example                   0 bytes
✅ app/main.py                     5749 bytes
✅ app/requirements.txt             205 bytes

❌ Some files missing — re-run the cells above

What Render does with these files on Day 28:
  1. Reads your Dockerfile
  2. Runs docker build automatically
  3. Runs docker run with your environment variables
  4. Gives you a public URL
  5. Auto-redeploys on every git push to main


## Day 27 Summary

**What we built:**
- `Dockerfile` — production-ready, with layer caching optimized, health check, correct host binding
- `.dockerignore` — excludes secrets, bytecode, notebooks, git history
- `docker-compose.yml` — FastAPI + ChromaDB orchestrated together with named volumes
- `.env.example` — documents required secrets without exposing them

**The three rules that matter forever:**

1. **Layer order:** requirements before code — cache the slow pip install
2. **Secrets:** never in Dockerfile or docker-compose.yml — always from environment
3. **Host binding:** always `--host 0.0.0.0` — default localhost is unreachable outside container

**Why we can't docker build on Kaggle:**
Kaggle doesn't expose the Docker daemon — you'd need root access to the host machine.
Render reads your Dockerfile from GitHub and builds it in their infrastructure.
That's why the files we wrote today ARE the deployment — push them, Render does the rest.

**What's next — Day 28:**
Push everything to a dedicated GitHub repo.
Connect it to Render.
Set environment variables in Render's dashboard.
Get a live public URL at `https://your-app.onrender.com`.
The Swagger UI at `/docs` will finally be accessible from any browser.

In [7]:
# Update requirements.txt with pinned versions for reproducible builds
requirements = """# FastAPI stack
fastapi==0.110.0
uvicorn[standard]==0.27.1
pydantic==2.6.4

# RAG pipeline
langchain==0.1.16
langchain-groq==0.1.3
langchain-chroma==0.1.0
chromadb==0.4.24
sentence-transformers==2.7.0
rank-bm25==0.2.2

# Utilities
python-dotenv==1.0.1
httpx==0.27.0
"""

with open("app/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt updated with pinned versions")
print()
print("Why pin versions in production:")
print("  Without pins: pip install gets latest → breaks when library updates")
print("  With pins:    identical environment every build → reproducible")
print()
print("Why >= in development, == in production:")
print("  Development: flexibility to get fixes")
print("  Production:  stability over everything")

✅ requirements.txt updated with pinned versions

Why pin versions in production:
  Without pins: pip install gets latest → breaks when library updates
  With pins:    identical environment every build → reproducible

Why >= in development, == in production:
  Development: flexibility to get fixes
  Production:  stability over everything


In [8]:
env_example = """# Copy this file to .env and fill in your values
# NEVER commit .env to git — only commit .env.example

# Groq API Key — get from console.groq.com
GROQ_API_KEY=your_groq_api_key_here

# ChromaDB connection (used when running via docker-compose)
CHROMA_HOST=chromadb
CHROMA_PORT=8001

# App settings
APP_ENV=production
LOG_LEVEL=warning
"""

with open("app/.env.example", "w") as f:
    f.write(env_example)

print("✅ .env.example written")

✅ .env.example written


In [9]:
import os

files = {
    "Dockerfile": "Dockerfile",
    ".dockerignore": ".dockerignore",
    "docker-compose.yml": "docker-compose.yml",
    "app/.env.example": "app/.env.example",
    "app/main.py": "app/main.py",
    "app/requirements.txt": "app/requirements.txt",
}

print("=" * 55)
print("Docker artifact verification")
print("=" * 55)

all_good = True
for name, path in files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "✅" if exists else "❌"
    print(f"{status} {name:<30} {size:>5} bytes")
    if not exists:
        all_good = False

print()
if all_good:
    print("✅ All Docker artifacts present — ready for Day 28")
else:
    print("❌ Some files missing")

Docker artifact verification
✅ Dockerfile                      3245 bytes
✅ .dockerignore                    339 bytes
✅ docker-compose.yml              2303 bytes
✅ app/.env.example                 340 bytes
✅ app/main.py                     5749 bytes
✅ app/requirements.txt             266 bytes

✅ All Docker artifacts present — ready for Day 28
